In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import re
import pandas as pd

tokenizer = AutoTokenizer.from_pretrained("njlr/cs180-project")
model = AutoModelForSequenceClassification.from_pretrained("njlr/cs180-project")
model.eval()  # Set model to eval mode

def clean_text(s: str):
    s = re.sub(r'[“”]', '"', s)
     
    # Replace all curly single quotes with '
    s = re.sub(r"[‘’]", "'", s)
    
    # Replace en dash and em dash with hyphen
    s = re.sub(r"[–—]", "-", s)

    # Only retain alphanumeric, whitespace characters, single and double quotes, and hyphens
    s = re.sub(pattern=rf"[^a-zA-Z0-9\s\-\'\"]", repl="", string=s, flags=re.IGNORECASE)

    # Remove extra whitespaces
    s = re.sub(pattern=r"\s+", repl=" ", string=s).strip()

    return s

def preprocess(text: str):
    return clean_text(text)

with open('demo_input.txt', 'r') as f:
    new_data = f.read().split('\n')

df_test = pd.DataFrame({'text' : new_data })
df_test["cleaned"] = df_test["text"].apply(preprocess)

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    model.config.id2label = {0: "Risk", 1: "Neutral", 2: "Opportunity"}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1)
        confidence, predicted_class = torch.max(probs, dim=1)
        label = model.config.id2label[predicted_class.item()]
    return label, confidence.item()

df_test[['predicted_label', 'confidence']] = df_test['cleaned'].apply(
    lambda x: pd.Series(predict(x))
)

for _, row in df_test.iterrows():
    print(f"Text: {row['text']}\nPredicted Label: {row['predicted_label']}\nConfidence: {row['confidence']}\n")

Text: 0                    Climate change is a global issue.
1             Green initiatives combat climate change.
2                     Climate change affects everyone.
3                      Global warming is not alarming.
4    For example, the E-FACE fund, which was create...
5    There are many projections about the future gr...
Name: text, dtype: object
Predicted Label: 0           Risk
1    Opportunity
2           Risk
3           Risk
4    Opportunity
5           Risk
Name: predicted_label, dtype: object
Confidence: 0    0.999579
1    0.995808
2    0.999009
3    0.999507
4    0.999769
5    0.999781
Name: confidence, dtype: float64

Text: 0                    Climate change is a global issue.
1             Green initiatives combat climate change.
2                     Climate change affects everyone.
3                      Global warming is not alarming.
4    For example, the E-FACE fund, which was create...
5    There are many projections about the future gr...
Name: text, dtyp